# 社會網絡分析與地理應用 第十五週作業

資訊工程學系 三年級 吳佳泰 S1254059

## 1. 請蒐集並整理台北捷運 OD 資料（挑選某一天）以及站點的經緯度座標，然後以 Flowmap City 平台繪製流動地圖，並且分享網址

https://app.flowmap.city/public/f6c716f2-74b6-4512-bd79-4ea8e30837a7

## 2. 呈上題，請利用 gephi lite 或是 pyvis 製作無經緯度座標的台北捷運 OD 網絡

https://taitaiwu.github.io/python/Social_Network_Analysis_and_Geographic_Applications/homework_w15_Q2.html

In [4]:
import csv
import networkx as nx

from collections import defaultdict
from pyvis.network import Network

# 1. Load OD data
pair_weight = defaultdict(int)
node_total = defaultdict(int) 

with open("homework_w15_data/flows.csv", encoding='utf-8') as f:
    for row in csv.DictReader(f):
        o, d, c = row['origin'], row['dest'], int(row['count'])

        if (o == d):
            continue

        key = tuple(sorted((o, d)))
        pair_weight[key] += c

for (a, b), w in pair_weight.items():
    node_total[a] += w
    node_total[b] += w

all_nodes = set(node_total.keys())

# 2. Select edges
edge_pairs = {pair for pair, w in pair_weight.items() if w >= 700}  # threshold can be changed
best_pair_for_node = {}

for (a, b), w in pair_weight.items():
    for n in (a, b):
        if ((n not in best_pair_for_node) or (w > pair_weight[best_pair_for_node[n]])):
            best_pair_for_node[n] = (a, b)

covered = set()

for a, b in edge_pairs:
    covered.add(a)
    covered.add(b)

for n in all_nodes - covered:
    edge_pairs.add(best_pair_for_node[n])

# 3. Build the graph
G = nx.Graph()
for a, b in edge_pairs:
    G.add_edge(a, b, weight=pair_weight[(a, b)])

print(f"Nodes: {G.number_of_nodes()} / {len(all_nodes)}  Edges: {G.number_of_edges()}")

# 4. Community detection (reveals clusters that aren't visible on a geo map)
communities = nx.community.greedy_modularity_communities(G, weight='weight')
community_of = {}

for i, com in enumerate(communities):
    for node in com:
        community_of[node] = i

print(f"Communities: {len(communities)}")

PALETTE = [
    "#e6194b", "#3cb44b", "#ffe119", "#4363d8", "#f58231",
    "#911eb4", "#46f0f0", "#f032e6", "#bcf60c", "#fabebe",
    "#008080", "#e6beff", "#9a6324", "#fffac8", "#800000",
    "#aaffc3", "#808000", "#ffd8b1", "#000075", "#a9a9a9",
]

# 5. Build pyvis network
net = Network(height="900px", width="100%", bgcolor="#1a1a2e", font_color="white")
net.force_atlas_2based(gravity=-50, central_gravity=0.01, spring_length=100, spring_strength=0.08, damping=0.4)

max_total = max(node_total[n] for n in G.nodes())
max_w = max(d["weight"] for _, _, d in G.edges(data=True))

for node in G.nodes():
    total = node_total[node]
    size = 8 + 32 * (total / max_total)
    color = PALETTE[community_of[node] % len(PALETTE)]
    net.add_node(node, label=node, size=size, color=color, title=f"{node}<br>對外總運量: {total:,} 人次/日")

for a, b, d in G.edges(data=True):
    w = d["weight"]
    width = 1 + 7 * (w / max_w)
    alpha = 0.15 + 0.65 * (w / max_w)
    net.add_edge(a, b, value=w, width=width, color=f"rgba(200,200,200,{alpha:.2f})",  title=f"{a} <-> {b}: {w:,} 人次/日")

net.show_buttons(filter_=["physics"])
net.write_html("homework_w15_Q2.html")
print("homework_w15_Q2.html is created.")


Nodes: 118 / 118  Edges: 580
Communities: 7
homework_w15_Q2.html is created.
